# FactLedger extractor

`load(path) -> documents, units` over raw files and nothing else. The extractor sniffs the
format from the bytes and writes document and unit JSON in the shapes of SCHEMA.md; the
rules it follows are in BUILD.md. Built one block at a time. Inputs: the public raw dataset
and the private papers dataset, both attached to this notebook.


In [ ]:
# Block 1: inputs and integrity.
# Mount both datasets, count files per folder, and check every file's sha256 against the
# folder manifest. The manifests are used here only to prove the Kaggle copies are the bytes
# that were uploaded; the extractor itself never reads them.
import hashlib
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

def mount(slug):
    """Kaggle mounts inputs at /kaggle/input/<slug> or, in newer sessions,
    /kaggle/input/datasets/<owner>/<slug>. Take whichever exists."""
    for candidate in (Path("/kaggle/input") / slug, Path("/kaggle/input/datasets/jhffmn") / slug):
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError(slug)


RAW = mount("it494-narrative-corpora-raw")
PAPERS = mount("it494-reference-papers")


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def check(folder, rows_key):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    rows = manifest[rows_key]
    on_disk = {p.name for p in folder.iterdir() if p.name not in ("manifest.json", "LICENSE")}
    listed = {r["file"] for r in rows}
    # Kaggle inputs are a network filesystem: one file at a time, 19,206 files take tens of
    # minutes; 32 concurrent reads take about a minute.
    with ThreadPoolExecutor(max_workers=32) as pool:
        digests = list(pool.map(sha256, [folder / r["file"] for r in rows]))
    bad = [r["file"] for r, d in zip(rows, digests) if d != r["sha256"]]
    print(f"{folder.name:<28} files {len(on_disk):>6}  listed {len(listed):>6}"
          f"  mismatched {len(bad)}  unlisted {len(on_disk - listed)}  missing {len(listed - on_disk)}")
    return bad


# The three literature manifests keep their original "works" key; the unpacked folders
# and the papers use "files".
for name, key in [("oz", "works"), ("holmes", "works"), ("greek", "works"),
                  ("graphrag-bench", "files"), ("longmemeval", "files")]:
    check(RAW / name, key)
check(PAPERS, "files")


In [ ]:
# Block 2: file type, then raw text.
#
# Two steps, by bytes only. Nothing here decides what the text is about; that is the model's
# job later.
#   1. file_kind(data): look at the first bytes and name the container: pdf, json, or text.
#   2. to_text(path): turn the container into one string, the document text.
#        pdf  -> the text layer, page by page (PyMuPDF, the one dependency)
#        json -> if it holds chat turns, one "role: content" block per turn under a header
#                of the session id and dates. We also keep where each turn starts and ends
#                in that string, so a chat can be cut into units without a model.
#        text -> the bytes decoded as UTF-8, unchanged
import json

try:
    import pymupdf
except ImportError:
    import subprocess
    import sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf"], check=True)
    import pymupdf


def file_kind(data):
    if data.startswith(b"%PDF-"):
        return "pdf"
    if data.lstrip()[:1] in (b"{", b"["):
        return "json"
    return "text"


def pdf_text(data):
    doc = pymupdf.open(stream=data, filetype="pdf")
    return "\n".join(page.get_text() for page in doc)


def chat_turns(obj):
    """The list of {role, content} turns inside a chat JSON, wherever it sits; None if absent."""
    if isinstance(obj, list) and obj and all(isinstance(t, dict) and "role" in t and "content" in t for t in obj):
        return obj
    if isinstance(obj, dict):
        for value in obj.values():
            found = chat_turns(value)
            if found:
                return found
    return None


def chat_text(obj, turns):
    """Header lines, a blank line, then 'role: content' per turn. Returns the text and the
    (start, end) of each turn inside it."""
    header = [f"session_id: {obj['session_id']}"] if "session_id" in obj else []
    header += [f"date: {d}" for d in obj.get("dates", [])]
    text = "\n".join(header) + "\n\n"
    spans = []
    for turn in turns:
        start = len(text)
        text += f"{turn['role']}: {turn['content']}\n\n"
        spans.append((start, len(text)))
    return text, spans


def to_text(path):
    data = path.read_bytes()
    doc = {"path": str(path), "sha256": hashlib.sha256(data).hexdigest(),
           "kind": file_kind(data), "text": "", "turns": None, "dates": []}
    if doc["kind"] == "pdf":
        doc["text"] = pdf_text(data)
    elif doc["kind"] == "json":
        obj = json.loads(data)
        turns = chat_turns(obj)
        if turns is None:
            doc["kind"], doc["text"] = "text", data.decode("utf-8", errors="replace")
        else:
            doc["kind"] = "chat"
            doc["text"], doc["turns"] = chat_text(obj, turns)
            doc["dates"] = list(obj.get("dates", []))
    else:
        doc["text"] = data.decode("utf-8", errors="replace")
    return doc


# One of each, to see the shape.
for path in [RAW / "oz" / "01_55.txt", RAW / "graphrag-bench" / "Novel-30752.txt",
             RAW / "longmemeval" / "sharegpt_yywfIrx_0.json", RAW / "longmemeval" / "001cefa7_2.json",
             PAPERS / "edge2024-graphrag.pdf"]:
    d = to_text(path)
    turns = len(d["turns"]) if d["turns"] else "-"
    print(f"{path.name:<26} {d['kind']:<5} {len(d['text']):>8,} chars  turns {turns:>3}  dates {d['dates']}")
    print("    " + repr(d["text"][:70]))


In [ ]:
# Block 3: the model call, and the check that a quoted string exists in the raw text.
import json
import time

import requests
from kaggle_secrets import UserSecretsClient

MODEL = "gpt-5.6-luna"
RETRY = "gpt-5.6-terra"
PRICE = {"gpt-5.6-luna": (0.20, 1.20), "gpt-5.6-terra": (2.00, 12.00)}   # $ per M tokens in, out
SPEND_STOP = 8.00                                                        # dollars; the run halts past this
KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
calls = []


def spend():
    return sum(c["cost"] for c in calls)


def generate(prompt, model=MODEL, effort="low"):
    """One JSON-mode call; the reply parsed, the cost logged. The API rejects temperature.
    A timeout, connection error, 429, or 5xx is retried three times with a pause; any other
    non-200 raises with the response body."""
    if spend() >= SPEND_STOP:
        raise RuntimeError(f"spending stop: ${spend():.2f}")
    t0 = time.time()
    for attempt in range(3):
        try:
            r = requests.post("https://api.openai.com/v1/chat/completions",
                              headers={"Authorization": f"Bearer {KEY}"}, timeout=300,
                              json={"model": model, "reasoning_effort": effort,
                                    "response_format": {"type": "json_object"},
                                    "messages": [{"role": "user", "content": prompt}]})
        except (requests.Timeout, requests.ConnectionError) as e:
            if attempt == 2:
                raise
            time.sleep(15 * (attempt + 1))
            continue
        if r.status_code == 429 or r.status_code >= 500:
            if attempt == 2:
                raise RuntimeError(f"OpenAI {r.status_code}: {r.text}")
            time.sleep(15 * (attempt + 1))
            continue
        if r.status_code != 200:
            raise RuntimeError(f"OpenAI {r.status_code}: {r.text}")
        break
    body = r.json()
    u, (p_in, p_out) = body["usage"], PRICE[model]
    calls.append({"model": body["model"], "in": u["prompt_tokens"], "out": u["completion_tokens"],
                  "seconds": round(time.time() - t0, 1),
                  "cost": (u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out) / 1e6})
    return json.loads(body["choices"][0]["message"]["content"])


NEAR = 100   # a quote's next line must begin within this many characters of the previous one


def owns_its_line(text, at, end, closed):
    """The match begins its line (only whitespace before it) and either ends it (nothing but
    punctuation or whitespace after) or is itself closed by punctuation, as a run-in heading
    like 'Abstract.' or 'Example 2.2.' is. A heading cannot match inside a sentence, and a
    title that merely opens a sentence does not count."""
    line_start = text.rfind("\n", 0, at) + 1
    line_end = text.find("\n", end)
    line_end = len(text) if line_end < 0 else line_end
    begins = not text[line_start:at].strip()
    ends = not any(c.isalnum() for c in text[end:line_end])
    return begins and (ends or closed)


def find(doc, quote, start=0, skip=None, whole_line=True):
    """Character position of the quote at or after start, case ignored: its lines together
    (each following line within NEAR characters of the previous), else its first line alone
    (the chapter breaker), else its last line alone (the title), else, for a heading in a
    document that has lines, the first three words of its first line at the start of a line
    (a scan's misprint later in the line, which the model copies corrected, must not lose the
    heading). At least three characters, not glued to a letter or digit on either side, not
    inside the skipped span (the contents list), and, for a structural quote in a document
    that has lines, owning its line. None if absent."""
    text = doc["lower"]
    lines = [p.strip().lower() for p in (quote or "").split("\n") if p.strip()]
    attempts = [lines, lines[:1], lines[-1:]] if len(lines) > 1 else [lines]
    opening = " ".join(lines[0].split()[:3]) if lines else ""
    if whole_line and doc["has_lines"] and len(lines[0].split()) > 3:
        attempts.append([opening])
    for parts in attempts:
        if not parts or len("".join(parts)) < 3:
            continue
        at = text.find(parts[0], start)
        while at >= 0:
            end, ok = at + len(parts[0]), True
            for part in parts[1:]:
                nxt = text.find(part, end, end + NEAR + len(part))
                if nxt < 0:
                    ok = False
                    break
                end = nxt + len(part)
            before = text[at - 1] if at > 0 else " "
            after = text[end] if end < len(text) else " "
            inside = skip is not None and skip[0] <= at < skip[1]
            closed = parts == [opening] or not parts[-1][-1].isalnum()   # the opening words need only begin their line
            placed = not (whole_line and doc["has_lines"]) or owns_its_line(text, at, end, closed)
            if ok and not before.isalnum() and not after.isalnum() and not inside and placed:
                return at
            at = text.find(parts[0], at + 1)
    return None


print(f"model {MODEL}, retry {RETRY}, spend stop ${SPEND_STOP:.2f}, key {'present' if KEY else 'MISSING'}")


In [ ]:
# Block 5: the question, and the text in overlapping chunks. Each chunk is about 14,000
# characters (3,500 tokens) cut at a line break, and the next one starts 12,000 characters
# on, so neighbours overlap by about 500 tokens and no heading is ever sliced in two. A chunk
# remembers its own (start, end) in the document, so whatever the model quotes from it is
# searched only inside that span; a heading quoted by both neighbours lands on one position.
CHUNK = 14_000
STRIDE = 12_000
CAP_WORDS = 4000

PROMPT = """Below is part %d of %d of one document, as raw text. Answer with JSON only. Every string you return must be copied from this part exactly, character for character, never paraphrased or corrected, because a program will search this part for it. A heading that spans two lines (a number on one line, its title on the next) is quoted with both lines and the line break between them; a heading that is only a number or numeral is quoted together with the line that follows it.

{
  "source_class": one of "canonical" (a published literary or classic work), "published" (a paper, article, or report), "authored" (a person's own material: notes, email, letters, drafts); null unless this is part 1,
  "title": the title as written, or null,
  "author": the author's name as written, or null,
  "date": {"quote": the complete line containing the date the work was written, published, or sent, "iso": "YYYY" or "YYYY-MM" or "YYYY-MM-DD"} or null. Not a transcription or ebook release date,
  "contents_start": the complete first line of a contents list in this part (its heading, or its first entry), or null,
  "contents_end": the complete last line of that contents list, or null,
  "body_start": the complete first line of the work itself, or null if the work does not begin in this part. Publisher notices, a contents list, and transcriber's or translator's notes are not part of the work; an author's own preface or introduction is,
  "end_matter_start": the complete first line of any end matter that follows the work (references, license, index, notes, advertisements), or null if this part has no such line. Notes or footnotes between divisions belong to the piece before them and are not end matter,
  "toc_count": the number of pieces the contents list gives, or null,
  "pieces": [{"marker": the complete heading line that begins a piece in this part, exactly as it appears where the piece begins, never as the contents list writes it, "title": a short title for the piece, such as "Chapter 1: The Cyclone" or "Abstract" or "Act II, Scene 1"}]
}

Pieces are the document's own divisions: chapters, acts and scenes, sections, dated entries, poems, stories. A paper's pieces are its sections, the abstract first. Aim for pieces under %d words; where a division is longer, use its next level down. A part with no piece beginning in it gets an empty pieces list.
%s
TEXT:
%s
"""

OUTLINE = """Below is the outline of one document, %s, %d characters long: every heading a first pass over its parts found, in order, with its character position, the number of words from it to the next line listed, and its first line. A line that some part called the start of the end matter is marked END?. Answer with JSON only, using positions copied from the list.

{
  "drop": [positions of listed lines that are not divisions of the work: a page header that repeats through the document, a line or verse number, a footnote number, a heading repeated for a translation on the facing page, an entry of a contents list],
  "end_matter_at": the position of the first listed line after which nothing of the work remains (references, license, index, notes, advertisements), or null if the work runs to the end of the document
}

OUTLINE:
%s
"""


def line_break_before(text, start, limit):
    """The position just after the last line break in [start, limit), else limit."""
    cut_at = text.rfind("\n", start, limit)
    return cut_at + 1 if cut_at > start else limit


def chunk_bounds(text):
    bounds, start = [], 0
    while True:
        end = len(text) if start + CHUNK >= len(text) else line_break_before(text, start, start + CHUNK)
        bounds.append((start, end))
        if end >= len(text):
            return bounds
        start = line_break_before(text, start, start + STRIDE)


def ask(text, bounds, i, feedback=""):
    start, end = bounds[i]
    return generate(PROMPT % (i + 1, len(bounds), CAP_WORDS, feedback, text[start:end]))


In [ ]:
# Block 6: from chunk answers to pieces.
#   locate     every quoted line of one chunk's answer found inside that chunk's span, or
#              listed as missing. A contents list quoted in the chunk is a span nothing is
#              located in. Headings are searched in order; one not found after the previous
#              is searched from the chunk start, so the model's list order does not matter.
#              Title, author, and date are phrases inside lines; everything else owns its line.
#   split_document  the text lower-cased once; every chunk asked, eight at a time; a chunk
#              with missing quotes is asked again with them fed back, twice at most; the
#              answers merged into one reply with the document position of everything found.
#              The body starts where the first chunk to say so says.
#   review_outline  one call over the whole outline, every located heading and every chunk's
#              end-matter claim in order with positions and word counts. A chunk cannot know
#              whether the work goes on after it or whether "21" is a heading or a line
#              number; the outline shows it. The model names the lines to drop and where the
#              end matter begins, by positions copied from the list.
#   cut        pieces as dicts (start, end, label) sorted by position: the gap before the
#              first heading is the opening piece; pieces after the end matter are dropped; a
#              piece under FLOOR words is a heading with no text of its own and folds into
#              the next.
from concurrent.futures import ThreadPoolExecutor

DOC_FIELDS = ("source_class", "title", "author", "date", "toc_count")   # first answer wins
FLOOR = 15                                                              # words; below it a piece is only a heading


def locate(doc, start, end, r):
    at, missing = {}, []

    def place(key, quote, from_pos, skip=None, whole_line=True):
        hit = find(doc, quote, from_pos, skip, whole_line)
        if hit is None or hit >= end:
            missing.append(quote)
            return None
        at[key] = hit
        return hit

    contents = None
    if r.get("contents_start") and r.get("contents_end"):
        a = place("contents_start", r["contents_start"], start)
        b = place("contents_end", r["contents_end"], a if a is not None else start)
        if a is not None and b is not None:
            contents = (a, b + len(r["contents_end"].strip()))   # the matched text, not the quote's padding
    for key in ("body_start", "end_matter_start"):
        if r.get(key):
            place(key, r[key], start, contents)
    pos = start
    for i, piece in enumerate(r.get("pieces") or []):
        hit = find(doc, piece.get("marker"), pos, contents)
        if hit is None or hit >= end:
            hit = place(f"piece {i}", piece.get("marker"), start, contents)
        else:
            at[f"piece {i}"] = hit
        if hit is not None:
            pos = hit + 1
    for quote in (r.get("title"), r.get("author"), (r.get("date") or {}).get("quote")):
        if quote:
            hit = find(doc, quote, start, whole_line=False)
            if hit is None or hit >= end:
                missing.append(quote)
    return at, missing


def split_document(doc, attempts=2):
    text = doc["text"]
    lowered = text.lower()
    doc["lower"] = lowered if len(lowered) == len(text) else text   # a lower-casing that changes length would shift offsets
    doc["has_lines"] = text.count("\n") >= len(text) / 500
    bounds = chunk_bounds(text)

    def one(i):
        feedback = ""
        for attempt in range(attempts):
            r = ask(text, bounds, i, feedback)
            at, missing = locate(doc, bounds[i][0], bounds[i][1], r)
            if not missing:
                return r, at, missing, attempt + 1
            feedback = ("\nThese strings from a previous answer were not found in this part as complete lines, or are "
                        "too short to search for alone; quote a bare numeral together with the line that follows it. "
                        "Copy every string exactly as it appears, including misprints, stray letters, and scanning "
                        "errors, which must stay as they stand: " + json.dumps(missing[:20], ensure_ascii=False) + "\n")
        return r, at, missing, attempts

    with ThreadPoolExecutor(max_workers=8) as pool:
        answers = list(pool.map(one, range(len(bounds))))

    reply, at, missing, tries, ends = {"pieces": [], "chunks": len(bounds)}, {}, [], 1, {}
    for r, chunk_at, chunk_missing, chunk_tries in answers:
        for key in DOC_FIELDS:
            if reply.get(key) is None and r.get(key) is not None:
                reply[key] = r[key]
        for key in ("body_start", "contents_start", "contents_end"):
            if key in chunk_at and key not in at:
                at[key], reply[key] = chunk_at[key], r[key]
        if "end_matter_start" in chunk_at:
            ends[chunk_at["end_matter_start"]] = r["end_matter_start"]
        for i, piece in enumerate(r.get("pieces") or []):
            if f"piece {i}" in chunk_at:
                at[f"piece {len(reply['pieces'])}"] = chunk_at[f"piece {i}"]
            reply["pieces"].append(piece)
        missing += chunk_missing
        tries = max(tries, chunk_tries)
    review_outline(doc, reply, at, ends)
    return reply, at, missing, tries


def review_outline(doc, reply, at, ends):
    text = doc["text"]
    rows = {}                                            # position -> indices of the pieces located there
    for i in range(len(reply["pieces"])):
        if f"piece {i}" in at:
            rows.setdefault(at[f"piece {i}"], []).append(i)
    positions = sorted(set(rows) | set(ends))
    if not positions:
        return
    first, lines = {}, []
    for pos, nxt in zip(positions, positions[1:] + [len(text)]):
        line_end = text.find("\n", pos)
        first[pos] = text[pos:line_end if line_end >= 0 else len(text)].strip()[:80]
        lines.append(f"{pos:>9} {len(text[pos:nxt].split()):>7} w  {'END?' if pos in ends else '    '}  {first[pos]}")
    r = generate(OUTLINE % (reply.get("title") or "untitled", len(text), "\n".join(lines)))
    drop = {int(x) for x in (r.get("drop") or []) if str(x).isdigit()} & set(rows)
    for pos in drop:
        for i in rows[pos]:
            del at[f"piece {i}"]
    end = r.get("end_matter_at")
    if end in positions:
        at["end_matter_start"] = end
        reply["end_matter_start"] = ends.get(end) or reply["pieces"][rows[end][0]]["marker"]
    elif end is None:
        reply["end_matter_start"] = None
    elif ends:                                           # an answer off the list: the earliest claim, as before
        at["end_matter_start"] = min(ends)
        reply["end_matter_start"] = ends[min(ends)]
    reply["review"] = {"dropped": [first[pos] for pos in sorted(drop)], "claims": len(ends),
                       "end_matter_at": at.get("end_matter_start")}


def cut(text, reply, at):
    end = at.get("end_matter_start", len(text))
    found = {}
    for i, p in enumerate(reply["pieces"]):
        if f"piece {i}" in at:
            found[at[f"piece {i}"]] = " ".join(str(p.get("title") or p.get("marker")).split())
    kept = [(s, label) for s, label in sorted(found.items()) if s < end]
    body_start = at.get("body_start", kept[0][0] if kept else 0)
    pieces = []
    if kept and text[body_start:kept[0][0]].strip():
        pieces.append({"start": body_start, "end": kept[0][0], "label": "opening"})
    bounds = [s for s, _ in kept] + [end]
    pieces += [{"start": s, "end": e, "label": label} for (s, label), e in zip(kept, bounds[1:])]
    if not kept:
        pieces.append({"start": body_start, "end": end, "label": reply.get("title") or "whole"})
    folded = []
    for p in pieces:
        if folded and len(text[folded[-1]["start"]:folded[-1]["end"]].split()) < FLOOR:
            prev = folded.pop()
            p = {"start": prev["start"], "end": p["end"], "label": f"{prev['label']} / {p['label']}"}
        folded.append(p)
    if len(folded) > 1 and len(text[folded[-1]["start"]:folded[-1]["end"]].split()) < FLOOR:
        last = folded.pop()
        folded[-1] = {**folded[-1], "end": last["end"]}
    return folded, len(found) - len(kept)


In [ ]:
# Block 7: every text and PDF in the dataset, resumable. Each finished document is appended
# to splits.jsonl (reply, positions, pieces, cost). On a rerun, documents that came out clean
# are skipped and flagged ones are tried again; the last record per document wins. Chats
# need no model call and are handled at export. Full detail is printed for every document.
SPLITS = Path("/kaggle/working/splits.jsonl")
LOG = Path("/kaggle/working/splits.log")
done = set()
if SPLITS.exists():
    for line in SPLITS.read_text(encoding="utf-8").split("\n"):      # never splitlines(): text can hold U+2028 or form feeds
        if line:
            record = json.loads(line)
            if not record["missing"]:
                done.add(record["sha256"])


def show(doc, record):
    """The full log entry for one document: header, metadata, body span, every piece with
    its position, word count, label, and opening words, then what the review dropped and
    anything missing or dropped. Printed and appended to splits.log."""
    text, reply, pieces = doc["text"], record["reply"], record["pieces"]
    body_start, body_end = pieces[0]["start"], pieces[-1]["end"]
    date = reply.get("date") or {}
    lines = [
        f"{Path(record['path']).name}  {'FLAGGED' if record['missing'] else 'ok'}  pieces {len(pieces)}"
        f"  dropped {record['dropped']}  tries {record['tries']}  toc {reply.get('toc_count')}"
        f"  chunks {reply.get('chunks')}  ${record['cost']:.3f}",
        f"    class {reply.get('source_class')} | title {reply.get('title')!r} | author {reply.get('author')!r}"
        f" | date {date.get('iso')} from {date.get('quote')!r}",
        f"    body {body_start}-{body_end} of {len(text)} ({1 - (body_end - body_start) / max(1, len(text)):.0%} outside)"
        f" | body_start {reply.get('body_start')!r} | end matter {reply.get('end_matter_start')!r}",
        f"    contents {reply.get('contents_start')!r} .. {reply.get('contents_end')!r}"
        f" at {record['at'].get('contents_start')}-{record['at'].get('contents_end')}",
    ]
    for p in pieces:
        snippet = " ".join(text[p["start"]:p["start"] + 90].split())[:60]
        lines.append(f"    {p['start']:>8} {len(text[p['start']:p['end']].split()):>6} w  {p['label'][:42]:<42} | {snippet}")
    review = reply.get("review") or {}
    if review.get("dropped"):
        lines.append(f"    review dropped {len(review['dropped'])} of the located lines ({review['claims']} end-matter claims): "
                     + "; ".join(review["dropped"])[:300])
    lines += [f"    MISSING {quote!r}" for quote in record["missing"]]
    if record["dropped"]:
        lines.append(f"    {record['dropped']} piece(s) after the end matter dropped")
    entry = "\n".join(lines) + "\n"
    print(entry)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(entry + "\n")


paths = sorted(RAW.glob("oz/*.txt")) + sorted(RAW.glob("holmes/*.txt")) + sorted(RAW.glob("greek/*.txt")) \
      + sorted(RAW.glob("graphrag-bench/*.txt")) + sorted(PAPERS.glob("*.pdf"))

for path in paths:
    doc = to_text(path)
    if doc["sha256"] in done:
        continue
    before = spend()
    try:
        reply, at, missing, tries = split_document(doc)
        pieces, dropped = cut(doc["text"], reply, at)
    except Exception as e:                      # one document must not end a two-hour run
        reply, at, tries, dropped = {"pieces": [], "chunks": None}, {}, 0, 0
        missing = [f"run error: {type(e).__name__}: {str(e)[:200]}"]
        pieces = [{"start": 0, "end": len(doc["text"]), "label": "whole document"}]
    record = {"path": str(path), "sha256": doc["sha256"], "reply": reply, "at": at, "missing": missing,
              "tries": tries, "pieces": pieces, "dropped": dropped, "cost": round(spend() - before, 4)}
    with SPLITS.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    done.add(doc["sha256"])
    show(doc, record)
print(f"{len(done)} documents in {SPLITS.name}, ${spend():.2f} spent this session")


In [ ]:
# Block 8: replay the full log for every document already in splits.jsonl. No model calls.
# The last record per document wins. Pass a name fragment to see one document.
WANT = ""     # e.g. "03_348" for the Hesiod anthology; empty for everything

latest = {}
for line in SPLITS.read_text(encoding="utf-8").split("\n"):
    if line:
        record = json.loads(line)
        latest[record["sha256"]] = record
LOG.write_text("", encoding="utf-8")
for record in latest.values():
    if WANT in record["path"]:
        show(to_text(Path(record["path"])), record)
print(f"{len(latest)} documents, {sum(1 for r in latest.values() if r['missing'])} flagged, log rewritten to {LOG.name}")


In [ ]:
# Block 9: export in the schema, plus the receipt.
#   documents.jsonl   one row per document with its text (SCHEMA.md: doc_id, source_uri,
#                     sha256, title, author, source_class, text, ingested_at, occurred_at, loader)
#   units.jsonl       ranges into the document text (unit_id, doc_id, position, label, start,
#                     end, occurred_at, occurred_until)
#   split_plans.jsonl the model's reply, the positions found, what was missing, tries, cost
#   receipt.json      documents by kind, units, flags by kind, unknown authors, ambiguous
#                     dates, units with times, over-cap units, boilerplate share, cost
# Texts and PDFs come from splits.jsonl (last record per document wins); every unit carries
# the document's own date (publication, or when it was written) on both ends, null when the
# file names none. Chats are packed here with no model call: turns grouped up to CAP_WORDS,
# never a lone turn, a tail under a third of the cap merged back; one session date fills
# occurred_at on the document and on every unit as both ends, several dates leave them null
# and flag the document ambiguous.
from datetime import datetime, timezone

OUT = Path("/kaggle/working/export")
OUT.mkdir(parents=True, exist_ok=True)
LOADER = "factledger-extractor 0.1"
NOW = datetime.now(timezone.utc).isoformat(timespec="seconds")


def parse_time(value):
    """ISO 8601, or the 'yyyy/mm/dd (Dow) hh:mm' form; None when it is neither."""
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00")).isoformat()
    except (ValueError, AttributeError):
        pass
    parts = value.replace("/", " ").replace(":", " ").split()
    try:
        y, mo, d = int(parts[0]), int(parts[1]), int(parts[2])
        h, mi = (int(parts[-2]), int(parts[-1])) if len(parts) >= 5 else (0, 0)
        return datetime(y, mo, d, h, mi).isoformat()
    except (ValueError, IndexError):
        return None


def chat_units(text, turns, when):
    groups, current = [], []
    for span in turns:
        if len(current) >= 2 and len(text[current[0][0]:span[1]].split()) > CAP_WORDS:
            groups.append(current)
            current = []
        current.append(span)
    if current:
        if groups and (len(current) < 2 or len(text[current[0][0]:current[-1][1]].split()) < CAP_WORDS / 3):
            groups[-1].extend(current)
        else:
            groups.append(current)
    first, units = 1, []
    for g in groups:
        units.append({"start": g[0][0], "end": g[-1][1], "label": f"turns {first}-{first + len(g) - 1}",
                      "occurred_at": when, "occurred_until": when})
        first += len(g)
    return units


def unit_id(doc_id, text, start, end):
    return hashlib.sha256(f"{doc_id}\n{text[start:end]}".encode("utf-8")).hexdigest()


latest = {}
for line in SPLITS.read_text(encoding="utf-8").split("\n"):
    if line:
        record = json.loads(line)
        latest[record["sha256"]] = record

receipt = {"documents": 0, "units": 0, "by_kind": {}, "flagged": [], "unknown_author": 0, "ambiguous_date": 0,
           "units_with_time": 0, "over_cap_units": 0, "boilerplate_chars": 0, "total_chars": 0,
           "cost": round(sum(r["cost"] for r in latest.values()), 3),
           "session_dedupe_key": "session id over non-empty slots: 25,112 slots, 1,230 empty, 19,829 distinct ids of which 623 only ever appear empty, 19,206 sessions; 18,474 by role-and-content hash"}
docs_f = (OUT / "documents.jsonl").open("w", encoding="utf-8")
units_f = (OUT / "units.jsonl").open("w", encoding="utf-8")
plans_f = (OUT / "split_plans.jsonl").open("w", encoding="utf-8")

paths = sorted(RAW.glob("oz/*.txt")) + sorted(RAW.glob("holmes/*.txt")) + sorted(RAW.glob("greek/*.txt")) \
      + sorted(RAW.glob("graphrag-bench/*.txt")) + sorted(PAPERS.glob("*.pdf")) + sorted(RAW.glob("longmemeval/*.json"))
for path in paths:
    if path.name == "manifest.json":
        continue
    doc = to_text(path)
    text, doc_id = doc["text"], doc["sha256"]
    title = author = occurred = source_class = None
    flags = []
    if doc["kind"] == "chat":
        source_class = "record"
        dates = [parse_time(d) for d in doc["dates"]]
        if len(dates) == 1 and dates[0]:
            occurred = dates[0]
        elif len(dates) > 1:
            flags.append("ambiguous-date")
            receipt["ambiguous_date"] += 1
        units = chat_units(text, doc["turns"], occurred)
        plan = {"kind": "chat", "turns": len(doc["turns"]), "dates": doc["dates"]}
    else:
        record = latest.get(doc_id)
        if record is None:
            continue                                   # not yet split; run block 7 first
        reply = record["reply"]
        source_class, title, author = reply.get("source_class"), reply.get("title"), reply.get("author")
        occurred = (reply.get("date") or {}).get("iso")
        units = [{"start": p["start"], "end": p["end"], "label": p["label"], "occurred_at": occurred, "occurred_until": occurred}
                 for p in record["pieces"]]
        if record["missing"]:
            flags.append("split incomplete: " + "; ".join(repr(q) for q in record["missing"][:3]))
        plan = {"kind": doc["kind"], "reply": reply, "at": record["at"], "missing": record["missing"],
                "tries": record["tries"], "dropped": record["dropped"], "cost": record["cost"]}
        outside = units[0]["start"] + len(text) - units[-1]["end"]
        if outside > len(text) / 3:
            flags.append(f"{outside / len(text):.0%} of the text outside the units")
    if author is None:
        receipt["unknown_author"] += 1
    if source_class is None:
        flags.append("source_class unknown")
    rel = f"{RAW.name}/{path.relative_to(RAW).as_posix()}" if RAW in path.parents else f"{PAPERS.name}/{path.name}"
    docs_f.write(json.dumps({"doc_id": doc_id, "source_uri": rel, "sha256": doc_id, "title": title, "author": author,
                             "source_class": source_class, "text": text, "ingested_at": NOW, "occurred_at": occurred,
                             "loader": LOADER, "flags": flags}, ensure_ascii=False) + "\n")
    for position, u in enumerate(units):
        units_f.write(json.dumps({"unit_id": unit_id(doc_id, text, u["start"], u["end"]), "doc_id": doc_id, "position": position,
                                  "label": u["label"], "start": u["start"], "end": u["end"],
                                  "occurred_at": u["occurred_at"], "occurred_until": u["occurred_until"]}, ensure_ascii=False) + "\n")
        if len(text[u["start"]:u["end"]].split()) > CAP_WORDS:
            receipt["over_cap_units"] += 1
        if u["occurred_at"]:
            receipt["units_with_time"] += 1
    plans_f.write(json.dumps({"doc_id": doc_id, "source_uri": rel, **plan}, ensure_ascii=False) + "\n")
    receipt["documents"] += 1
    receipt["units"] += len(units)
    receipt["by_kind"][doc["kind"]] = receipt["by_kind"].get(doc["kind"], 0) + 1
    receipt["total_chars"] += len(text)
    receipt["boilerplate_chars"] += units[0]["start"] + (len(text) - units[-1]["end"])
    if flags:
        receipt["flagged"].append({"source_uri": rel, "flags": flags})

for f in (docs_f, units_f, plans_f):
    f.close()
receipt["boilerplate_share"] = round(receipt["boilerplate_chars"] / max(1, receipt["total_chars"]), 4)
(OUT / "receipt.json").write_text(json.dumps(receipt, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"documents {receipt['documents']}  units {receipt['units']}  by kind {receipt['by_kind']}")
print(f"flagged {len(receipt['flagged'])}  unknown author {receipt['unknown_author']}  ambiguous date {receipt['ambiguous_date']}"
      f"  units with time {receipt['units_with_time']}  over-cap units {receipt['over_cap_units']}"
      f"  boilerplate {receipt['boilerplate_share']:.1%}  model cost ${receipt['cost']:.2f}")
kinds = {}
for entry in receipt["flagged"]:
    for flag in entry["flags"]:
        kinds[flag.split(":")[0]] = kinds.get(flag.split(":")[0], 0) + 1
print("flags by kind:", kinds)
for entry in receipt["flagged"]:
    if "longmemeval/" not in entry["source_uri"]:
        print(f"    {entry['source_uri']}: {entry['flags']}")
